# Fundamental Agent FinBERT Training

This Colab notebook fine-tunes `ProsusAI/finbert` for the Fundamental Agent using:

`data/gold/fundamental_finetune_data.jsonl`

Expected output:

`models/fundamental_agent_finbert/`

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Set Project Paths

Make sure these files exist in Drive:

- `/content/drive/MyDrive/298A/agents/fundamental_finbert.py`
- `/content/drive/MyDrive/298A/data/gold/fundamental_finetune_data.jsonl`

In [ ]:
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/298A')
TRAIN_JSONL = PROJECT_DIR / 'data' / 'gold' / 'fundamental_finetune_data.jsonl'
SCRIPT_PATH = PROJECT_DIR / 'agents' / 'fundamental_finbert.py'
OUTPUT_DIR = PROJECT_DIR / 'models' / 'fundamental_agent_finbert'

print('PROJECT_DIR:', PROJECT_DIR)
print('TRAIN_JSONL:', TRAIN_JSONL, TRAIN_JSONL.exists())
print('SCRIPT_PATH:', SCRIPT_PATH, SCRIPT_PATH.exists())
print('OUTPUT_DIR:', OUTPUT_DIR)

## 3. Install Dependencies

In [ ]:
!pip install -q torch transformers datasets scikit-learn accelerate safetensors

## 4. Check GPU

In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 5. Verify Training Data

In [ ]:
import json
from collections import Counter

rows = []
with TRAIN_JSONL.open(encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

labels = Counter()
for row in rows:
    assistant = next(m for m in row['messages'] if m['role'] == 'assistant')
    payload = json.loads(assistant['content'])
    labels[payload['fundamental_risk_label']] += 1

print('examples:', len(rows))
print('labels:', dict(labels))
print(rows[0]['messages'][1]['content'][:500])

## 6. Verify Trainer Compatibility Patch

This catches the Colab error: `Trainer.__init__() got an unexpected keyword argument 'tokenizer'`.

In [ ]:
script_text = SCRIPT_PATH.read_text(encoding='utf-8')
has_patch = 'trainer_kwargs' in script_text and 'processing_class' in script_text
print('Trainer compatibility patch present:', has_patch)
if not has_patch:
    raise RuntimeError('Your Drive copy of fundamental_finbert.py is old. Re-upload the updated file before training.')

## 7. Train FinBERT

T4 GPU is enough. If training is slow, reduce epochs to `2` for a quick test.

In [ ]:
!python3 "{SCRIPT_PATH}" train \
  --train-jsonl "{TRAIN_JSONL}" \
  --output-dir "{OUTPUT_DIR}" \
  --epochs 4 \
  --batch-size 8

## 8. Verify Saved Model Files

In [ ]:
!ls -lh "{OUTPUT_DIR}"
!test -f "{OUTPUT_DIR}/config.json" && echo 'config.json found'
!test -f "{OUTPUT_DIR}/model.safetensors" && echo 'model.safetensors found'
!test -f "{OUTPUT_DIR}/tokenizer.json" && echo 'tokenizer.json found'
!test -f "{OUTPUT_DIR}/training_metrics.json" && echo 'training_metrics.json found'

## 9. Test Prediction

In [ ]:
!python3 "{SCRIPT_PATH}" predict AAPL --model-dir "{OUTPUT_DIR}"

## 10. Zip Model For VSCode

In [ ]:
!cd "{OUTPUT_DIR.parent}" && zip -qr fundamental_agent_finbert.zip fundamental_agent_finbert
!ls -lh "{OUTPUT_DIR.parent}/fundamental_agent_finbert.zip"

## 11. Download Zip

After downloading, unzip it locally into your project `outputs/` folder.

In [ ]:
from google.colab import files
files.download(str(OUTPUT_DIR.parent / 'fundamental_agent_finbert.zip'))